<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/Sankey_01/sankey1_master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sankey diagram 5
### Target Group → Target Class  →  Territorial reference point

#  Import libaries

In [1]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import plotly.colors

In [ ]:
# Airtable credentials
AIRTABLE_API_KEY = "din_nøgle_her"
base_id = 'apprKfEKZ2Ju74g9w'

# Workaround for limit issue with Airtable package
#api_key = 'patcGWLkQn1yXVi2Q.0fe39013f973b224df024ee97c41383368f28363519ccba57a768bf45a1ea8b0'
#base_id = 'appdBIB2W9qMw3efZ'

In [20]:
# Helper function to fetch data from Airtable
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [21]:
# Fetch data
tabel0 = fetch_airtable_data("tblzHR1WHYHA5MlwQ")  # Policy Source
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets (mellemtabel)
tabel2 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel3 = fetch_airtable_data("tblGwx3sWdVfvlMto")  # Target Class 

In [22]:
# Fjern rækker med manglende værdier

# For tabel0: behold kun rækker hvor både 'Policy Source' og 'Territorial reference point' er udfyldt
tabel0 = tabel0.dropna(subset=['Policy source','Territorial reference point'])

# For tabel1: behold kun rækker hvor både 'Target name', 'Target Group' og 'Target Class' er udfyldt
tabel1 = tabel1.dropna(subset=['Target name','Target Group','Target Class'])

# For tabel2: behold kun rækker hvor både 'Target Group' og 'Targets' er udfyldt
tabel2 = tabel2.dropna(subset=['Target Group', 'Targets'])

# For tabel3: behold kun rækker hvor både 'Name' og 'Targets' er udfyldt
tabel3 = tabel3.dropna(subset=['Name','Targets'])


In [23]:
# Policy Source
t0 = (
    tabel0.copy()
          .explode("Targets (policy targets)")
          .explode("Territorial reference point")
          .rename(columns={"Targets (policy targets)": "target_id"})
          [["id", "target_id", "Policy source", "Territorial reference point"]]   # kun de nødvendige
)

# Targets → Target Group
t1_groups = (
    tabel1.copy()
          .explode("Target Group")
          .explode("Policy Source")
          .rename(columns={"Target Group": "target_group_id"})
          [["id", "target_group_id", "Policy Source"]]   # behold kun det nødvendige
)

# Targets → Target Class
t1_classes = (
    tabel1.copy()
          .explode("Target Class")
          .explode("Policy Source")
          .rename(columns={"Target Class": "target_class_id"})
          [["id", "target_class_id", "Policy Source"]]   # behold kun det nødvendige
)

# Target Group tabel
t2 = (
    tabel2.copy()
          .explode("Targets")
          .rename(columns={"Targets": "target_id_from_group"})
          [["id", "Target Group", "target_id_from_group"]]
)

# Target Class tabel
t3 = (
    tabel3.copy()
          .explode("Targets")
          .rename(columns={"Targets": "target_id_from_class"})
          [["id", "Name", "target_id_from_class"]]
)


In [ ]:
# Filtrér direkte før registrering
# allowed_policies = [
    #"Technical Summary: Land Use and Climate Change",
    #"Convention on Biological Diversity",
    #"Common approach to integrating biodiversity and nature-based solutions for sustainable development into the United Nations policy and programme planning and delivery",
    #"European Green Deal",
    #"Common Agricultural Policy - Strategic plan 2023-2027",
    #"EU biodiversity strategy 2030",
    #"Aftale om et grønt Danmark (grøn trepart)",
    #"Mere, bedre og større natur i Danmark",
    #"Vandområdeplaner 2021-2027"
 #]
#t0 = t0[t0["Policy source"].isin(allowed_policies)]

#allowed_groups = ["Groundwater and water"]
#t2 = t2[t2["Target Group"].isin(allowed_groups)]

In [24]:
# Register in DuckDB
duckdb.register("tabel0", t0)              # Policy source → target_id
duckdb.register("tabel1a", t1_groups)      # target_id → target_group_id
duckdb.register("tabel1b", t1_classes)     # target_id → target_class_id
duckdb.register("tabel2", t2)              # target_group_id → target_id_from_group
duckdb.register("tabel3", t3)              # target_class_id → target_id_from_class


In [25]:
query = """
SELECT
    t2."Target Group"                AS target_group,
    t3."Name"                        AS target_class,
    t0."Territorial reference point" AS territorial_ref
FROM tabel2 AS t2
JOIN tabel1a AS g
  ON t2.target_id_from_group = g.id
JOIN tabel1b AS c
  ON g.id = c.id
JOIN tabel3 AS t3
  ON c.target_class_id = t3.id
JOIN tabel0 AS t0
  ON g."Policy Source" = t0.id
WHERE t0."Territorial reference point" IS NOT NULL
"""

results = duckdb.sql(query).df()
# print("Rækker i results:", len(results))
# print(results.head(10))


In [ ]:
# print("Antal rækker i resultatet:", len(results))
# results.head(10) 

In [26]:
# Forkort og saml labels
results['target_class'] = results['target_class'].apply(
    lambda x: x[:40] + '…' if isinstance(x, str) and len(x) > 40 else x
)

# Brug target_class direkte som "final_target"
results["final_target"] = results["target_class"]


#Udarbejd labels
group_labels = results['target_group']
class_labels = results['target_class']
territorial_labels = results['territorial_ref']
 

all_labels = pd.concat([group_labels, class_labels, territorial_labels])
unique_labels = pd.unique(all_labels)
label_to_index = {label: i for i, label in enumerate(unique_labels)}



In [27]:
# Link 1: target_group → target_class
links1 = pd.DataFrame({
    'source': results['target_group'].map(label_to_index).values,
    'target': results['target_class'].map(label_to_index).values,
    'value': 1
})

# Link 2: target_class → territorial_ref
links2 = pd.DataFrame({
    'source': results['target_class'].map(label_to_index).values,
    'target': results['territorial_ref'].map(label_to_index).values,
    'value': 1
})

# Combine both links
all_links = pd.concat([links1, links2], ignore_index=True)

# print("Antal links:", len(all_links))
# print(all_links.head())


In [29]:
# Brug Plotlys palette
node_colors = plotly.colors.qualitative.Plotly

# Tildel farver til unikke labels (noder)
color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(unique_labels)}

# Liste med farver i samme rækkefølge som unique_labels
node_colors_list = [color_map[label] for label in unique_labels]


# Funktion til at lysne farver (uden brug af gennemsigtighed)
def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

# Lysnet farve til hver link ud fra source-node
link_colors = [lighten(node_colors_list[src], factor=0.8) for src in all_links['source']]



# Trin 3: Sankey-diagram
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
        color=node_colors_list
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        color=link_colors
    )
)])
fig.update_layout(title_text="Target Group → Target Class → Territorial reference point", font_size=12, height=300, width=300)
fig.show()

KeyboardInterrupt: 

In [ ]:
# Gem som interaktiv HTML-fil
# fig.write_html("sankey_diagram_filtered.html")

# Download i Colab
# from google.colab import files
# files.download("sankey_diagram_filtered.html")